# STEP-004 & STEP-006 — Visual EDA (`fig01`–`fig05`), Augmentation Preview (`fig06`) & Auto-Push to GitHub

**Project:** AI-Generated Face Detection for Digital Identity Verification (e-KYC) and Fraud Prevention (`GROUP 08`)  
**Owner:** Member 2 — Khadijatul Kobra Keya (`0242310005101223`)  
**Dataset:** `kaggle:isthisdeception/ds0001-artifact-face-subset` (ArtiFact Face Subset, 50k images, MIT License)  
**Single Source of Truth:** `https://github.com/isthisdeception/computer-vission-course-work`

### One-Click Kaggle Checklist (Before clicking **Run All**)
1. **Add Input → Datasets:** Attach `isthisdeception/ds0001-artifact-face-subset`
2. **Session Options → Internet:** Turn **ON**
3. **Add-ons → Secrets:** Toggle **ON** `GITHUB_TOKEN` (must have `Contents: Read and write` on `computer-vission-course-work`)
4. Click **Run All** — generates `fig01`–`fig06` (300 DPI), `eda_summary.json`, `eda_generator_counts.csv`, `eda_report.md`, updates `figure_registry.csv` & `PROGRESS_TRACKER.md` (`GATE 2 PASS`), and pushes everything straight back to GitHub `main`!


In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.ndimage import laplace
from torchvision import transforms as T
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_URL = "https://github.com/isthisdeception/computer-vission-course-work.git"
BRANCH = "main"
GIT_USER_NAME = "Assaduzzaman"
GIT_USER_EMAIL = "md.assaduzzaman99@gmail.com"
SECRET_NAMES = ("GITHUB_TOKEN", "GH_TOKEN", "GITHUB_PAT", "GIT_TOKEN", "GITHUB_ACCESS_TOKEN", "github_token")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})


def load_github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in SECRET_NAMES:
            try:
                val = client.get_secret(name)
                if val and val.strip():
                    return val.strip(), f"Kaggle Secret '{name}'"
            except Exception:
                continue
    except ImportError:
        pass
    for name in SECRET_NAMES:
        val = os.environ.get(name, "").strip()
        if val:
            return val, f"env '{name}'"
    raise RuntimeError("GitHub token not found! Enable 'GITHUB_TOKEN' in Kaggle Add-ons -> Secrets.")


def run_masked(cmd, cwd=None, secret=None):
    res = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, check=False)
    mask = (lambda s: s.replace(secret, "***")) if secret else (lambda s: s)
    out = mask(res.stdout.strip())
    err = mask(res.stderr.strip())
    if out:
        print(out)
    if res.returncode != 0:
        raise RuntimeError(f"Command failed (rc={res.returncode}): {mask(' '.join(cmd))}\nSTDERR: {err}")
    return out


TOKEN, TOKEN_SRC = load_github_token()
print(f"[Setup] Loaded GitHub token from {TOKEN_SRC} (masked).")

REPO_DIR = Path("/kaggle/working/_repo_sync")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

AUTH_URL = REPO_URL.replace("https://", f"https://x-access-token:{TOKEN}@", 1)
run_masked(["git", "clone", "--depth", "1", "--branch", BRANCH, AUTH_URL, str(REPO_DIR)], secret=TOKEN)
run_masked(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR, secret=TOKEN)
run_masked(["git", "config", "user.name", GIT_USER_NAME], cwd=REPO_DIR, secret=TOKEN)
run_masked(["git", "config", "user.email", GIT_USER_EMAIL], cwd=REPO_DIR, secret=TOKEN)

sys.path.insert(0, str(REPO_DIR / "03_preprocessing"))
from transforms import (
    RandomJPEGCompression,
    denormalize_tensor,
    get_eval_transforms,
    get_train_transforms_robust,
)

# Auto-detect DS0001 root on Kaggle
CANDIDATE_ROOTS = [
    Path("/kaggle/input/datasets/isthisdeception/ds0001-artifact-face-subset"),
    Path("/kaggle/input/ds0001-artifact-face-subset"),
]
MOUNT_ROOT = next((c for c in CANDIDATE_ROOTS if c.exists()), None)
if MOUNT_ROOT is None and Path("/kaggle/input").exists():
    matches = [p for p in Path("/kaggle/input").rglob("*ds0001*") if p.is_dir()]
    MOUNT_ROOT = matches[0] if matches else None
assert MOUNT_ROOT is not None, "Attach Kaggle dataset 'isthisdeception/ds0001-artifact-face-subset'!"

DATA_ROOT = MOUNT_ROOT
for p in [MOUNT_ROOT, *MOUNT_ROOT.rglob("*")]:
    if p.is_dir() and (p / "real").is_dir() and (p / "fake").is_dir():
        DATA_ROOT = p
        break

FIG_DIR = REPO_DIR / "06_evaluation/figures/eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)
df_split = pd.read_csv(REPO_DIR / "02_data/splits/ds0001_split_v1.csv")
print(f"DATA_ROOT: {DATA_ROOT} | Loaded split rows: {len(df_split):,}")


In [ ]:
# 1. FIG01 — Class Balance, 70/15/15 Split Verification & 13-Generator Distribution
gen_counts = (
    df_split.groupby(["label_name", "generator", "split"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
gen_counts["total"] = gen_counts["train"] + gen_counts["val"] + gen_counts["test"]
gen_counts = gen_counts.sort_values(["label_name", "total"], ascending=[False, False]).reset_index(drop=True)
gen_counts.to_csv(REPO_DIR / "02_data/eda_generator_counts.csv", index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300, gridspec_kw={"width_ratios": [1, 1.45]})

# Panel A: Split counts by class
splits_order = ["train", "val", "test"]
real_vals = [int(((df_split["split"] == s) & (df_split["label"] == 0)).sum()) for s in splits_order]
fake_vals = [int(((df_split["split"] == s) & (df_split["label"] == 1)).sum()) for s in splits_order]
x = np.arange(len(splits_order))
w = 0.36
b1 = ax1.bar(x - w / 2, real_vals, width=w, label="Real (0)", color="#1f7a4c")
b2 = ax1.bar(x + w / 2, fake_vals, width=w, label="AI-Generated Fake (1)", color="#c0392b")
ax1.set_xticks(x)
ax1.set_xticklabels(["Train (70%)", "Val (15%)", "Test (15%)"])
ax1.set_ylabel("Image Count")
ax1.set_title("A. Leak-Free Stratified Split Balance (50,000 Total)", fontweight="bold")
ax1.legend()
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2, h + 350, f"{int(h):,}", ha="center", va="bottom", fontsize=8.5)
ax1.set_ylim(0, max(real_vals) * 1.15)

# Panel B: 13 Synthetic Generators
fake_df = gen_counts[gen_counts["label_name"] == "fake"].sort_values("total", ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(fake_df)))
bars = ax2.barh(fake_df["generator"], fake_df["total"], color=colors)
ax2.set_xlabel("Image Count")
ax2.set_title("B. Balanced Distribution Across 13 Synthetic Generators (25,000 Fake)", fontweight="bold")
for bar in bars:
    val = bar.get_width()
    ax2.text(val + 25, bar.get_y() + bar.get_height() / 2, f"{int(val):,}", va="center", fontsize=8.5)
ax2.set_xlim(0, fake_df["total"].max() * 1.15)

plt.tight_layout()
fig01_path = FIG_DIR / "fig01_class_generator_distribution.png"
plt.savefig(fig01_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig01_path.name)


In [ ]:
# 2. FIG02 — Visual Sample Grid: 4 Real Faces + 12 Distinct AI Generators (4x4)
real_4 = df_split[df_split["label"] == 0].sample(4, random_state=SEED)
all_fake_gens = sorted(df_split.loc[df_split["label"] == 1, "generator"].unique())
# Pick 12 diverse generators covering Diffusion, GANs, Transformers, Inpainting, 3D
selected_12 = [
    "stable_diffusion", "stylegan3", "stylegan2", "stylegan1",
    "taming_transformer", "gansformer", "projected_gan", "stargan",
    "sfhq", "face_synthetics", "lama", "mat",
]
fake_12 = pd.concat([
    df_split[df_split["generator"] == g].sample(1, random_state=SEED)
    for g in selected_12 if g in all_fake_gens
], ignore_index=True)
grid_df = pd.concat([real_4, fake_12], ignore_index=True)

fig, axes = plt.subplots(4, 4, figsize=(11.5, 12), dpi=300)
for ax, row in zip(axes.flat, grid_df.itertuples(index=False)):
    with Image.open(DATA_ROOT / row.rel_path) as im:
        ax.imshow(im.convert("RGB"))
    is_real = (row.label == 0)
    title_color = "#1b7f3b" if is_real else "#b42318"
    tag = "REAL" if is_real else f"FAKE: {row.generator}"
    ax.set_title(f"{tag}\n({row.image_id})", fontsize=8.5, color=title_color, fontweight="bold")
    ax.axis("off")

plt.suptitle("DS0001 Visual Sample Gallery — Real Human Faces vs. 12 Synthetic Generators", fontsize=13, fontweight="bold", y=0.995)
plt.tight_layout()
fig02_path = FIG_DIR / "fig02_sample_grid_real_vs_fake.png"
plt.savefig(fig02_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig02_path.name)


In [ ]:
# 3. Stratified Pixel, Lighting, Contrast, Sharpness/Blur & 2D FFT Analysis (n=4,000 sample)
SAMPLE_PER_CLASS = 2000
real_sample = df_split[df_split["label"] == 0].sample(SAMPLE_PER_CLASS, random_state=SEED)
# Balanced across all 13 fake generators (~154 each -> 2,000 total)
fake_parts = []
per_gen = SAMPLE_PER_CLASS // 13
for i, g in enumerate(sorted(df_split.loc[df_split["label"] == 1, "generator"].unique())):
    n_take = per_gen + (1 if i < (SAMPLE_PER_CLASS % 13) else 0)
    fake_parts.append(df_split[df_split["generator"] == g].sample(n_take, random_state=SEED))
sample_df = pd.concat([real_sample, *fake_parts], ignore_index=True)
print(f"Stratified EDA pixel sample size: {len(sample_df):,} (Real: {(sample_df['label']==0).sum()}, Fake: {(sample_df['label']==1).sum()})")

# Precompute azimuthal radial bins for 200x200 2D FFT
H, W = 200, 200
yy, xx = np.indices((H, W))
rr = np.sqrt((xx - W / 2.0) ** 2 + (yy - H / 2.0) ** 2).astype(int)
MAX_R = min(H, W) // 2
radial_counts = np.bincount(rr.ravel(), minlength=MAX_R + 1)[:MAX_R]


def compute_image_stats(row_tuple):
    rel_path, label, generator, image_id = row_tuple
    with Image.open(DATA_ROOT / rel_path) as im:
        arr = np.asarray(im.convert("RGB"), dtype=np.float32) / 255.0
    r_m, g_m, b_m = arr[:, :, 0].mean(), arr[:, :, 1].mean(), arr[:, :, 2].mean()
    luma = 0.299 * arr[:, :, 0] + 0.587 * arr[:, :, 1] + 0.114 * arr[:, :, 2]
    brightness = float(luma.mean())
    contrast = float(luma.std())
    sharpness = float(laplace(luma).var())  # Laplacian variance (higher = sharper, lower = blurrier)
    warmth = float(r_m - b_m)

    # 2D FFT log-magnitude spectrum (normalized zero-mean luma)
    f = np.fft.fftshift(np.fft.fft2(luma - brightness))
    mag = np.log1p(np.abs(f))
    radial_sum = np.bincount(rr.ravel(), weights=mag.ravel(), minlength=MAX_R + 1)[:MAX_R]
    radial_profile = radial_sum / np.maximum(radial_counts, 1)

    return {
        "image_id": image_id,
        "rel_path": rel_path,
        "label": int(label),
        "generator": generator,
        "brightness": brightness,
        "contrast": contrast,
        "sharpness_lap_var": sharpness,
        "r_mean": float(r_m),
        "g_mean": float(g_m),
        "b_mean": float(b_m),
        "warmth_r_minus_b": warmth,
        "fft_radial": radial_profile,
        "fft_2d": mag if generator in ("none", "stylegan2", "stable_diffusion") else None,
    }


tuples = list(zip(sample_df["rel_path"], sample_df["label"], sample_df["generator"], sample_df["image_id"]))
stats_rows = []
fft_accum = {"real": np.zeros((H, W), dtype=np.float64), "stylegan2": np.zeros((H, W), dtype=np.float64), "stable_diffusion": np.zeros((H, W), dtype=np.float64)}
fft_n = {"real": 0, "stylegan2": 0, "stable_diffusion": 0}
radial_by_group = {"Real (n=2000)": [], "All Fakes (n=2000)": [], "StyleGAN2": [], "Stable Diffusion": []}

with ThreadPoolExecutor(max_workers=8) as ex:
    for res in tqdm(ex.map(compute_image_stats, tuples), total=len(tuples), desc="Computing pixel & FFT stats"):
        rad = res.pop("fft_radial")
        mag2d = res.pop("fft_2d")
        stats_rows.append(res)
        if res["label"] == 0:
            radial_by_group["Real (n=2000)"].append(rad)
            if mag2d is not None:
                fft_accum["real"] += mag2d
                fft_n["real"] += 1
        else:
            radial_by_group["All Fakes (n=2000)"].append(rad)
            if res["generator"] == "stylegan2":
                radial_by_group["StyleGAN2"].append(rad)
                if mag2d is not None:
                    fft_accum["stylegan2"] += mag2d
                    fft_n["stylegan2"] += 1
            elif res["generator"] == "stable_diffusion":
                radial_by_group["Stable Diffusion"].append(rad)
                if mag2d is not None:
                    fft_accum["stable_diffusion"] += mag2d
                    fft_n["stable_diffusion"] += 1

df_stats = pd.DataFrame(stats_rows)
print(df_stats.groupby("label")[["brightness", "contrast", "sharpness_lap_var"]].agg(["mean", "std"]).round(4))


In [ ]:
# 4. FIG03 — Environmental Variations (Data-Driven Extremes: Lighting, Blur, Contrast, Color Cast)
selections = [
    ("Low-Light / Dark", df_stats.nsmallest(1, "brightness").iloc[0]),
    ("High-Key / Bright", df_stats.nlargest(1, "brightness").iloc[0]),
    ("Blurry / Soft Focus", df_stats.nsmallest(1, "sharpness_lap_var").iloc[0]),
    ("Ultra-Sharp Texture", df_stats.nlargest(1, "sharpness_lap_var").iloc[0]),
    ("Low Contrast / Hazy", df_stats.nsmallest(1, "contrast").iloc[0]),
    ("High Contrast / Shadow", df_stats.nlargest(1, "contrast").iloc[0]),
    ("Warm / Yellow Lighting", df_stats.nlargest(1, "warmth_r_minus_b").iloc[0]),
    ("Cool / Blue Lighting", df_stats.nsmallest(1, "warmth_r_minus_b").iloc[0]),
]

fig, axes = plt.subplots(2, 4, figsize=(13, 6.8), dpi=300)
for ax, (category, row) in zip(axes.flat, selections):
    with Image.open(DATA_ROOT / row["rel_path"]) as im:
        ax.imshow(im.convert("RGB"))
    cls = "REAL" if row["label"] == 0 else f"FAKE ({row['generator']})"
    ax.set_title(
        f"{category}\n{cls}\nLuma={row['brightness']:.2f} | BlurVar={row['sharpness_lap_var']:.4f}",
        fontsize=8,
        fontweight="bold",
    )
    ax.axis("off")

plt.suptitle("DS0001 Environmental Variations — Lighting, Sharpness/Blur, Contrast & Color Cast", fontsize=12.5, fontweight="bold")
plt.tight_layout()
fig03_path = FIG_DIR / "fig03_environmental_variations.png"
plt.savefig(fig03_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig03_path.name)


In [ ]:
# 5. FIG04 — Pixel Brightness, Contrast, Laplacian Sharpness & RGB Channel Stats (Real vs. Fake)
real_s = df_stats[df_stats["label"] == 0]
fake_s = df_stats[df_stats["label"] == 1]

fig, axes = plt.subplots(2, 2, figsize=(12.5, 9), dpi=300)

# Brightness
axes[0, 0].hist(real_s["brightness"], bins=40, alpha=0.55, density=True, label=f"Real (mean={real_s['brightness'].mean():.3f})", color="#1f7a4c")
axes[0, 0].hist(fake_s["brightness"], bins=40, alpha=0.55, density=True, label=f"Fake (mean={fake_s['brightness'].mean():.3f})", color="#c0392b")
axes[0, 0].set_title("A. Luminance (Brightness) Distribution", fontweight="bold")
axes[0, 0].set_xlabel("Mean Luminance [0, 1]")
axes[0, 0].set_ylabel("Density")
axes[0, 0].legend()

# Contrast
axes[0, 1].hist(real_s["contrast"], bins=40, alpha=0.55, density=True, label=f"Real (mean={real_s['contrast'].mean():.3f})", color="#1f7a4c")
axes[0, 1].hist(fake_s["contrast"], bins=40, alpha=0.55, density=True, label=f"Fake (mean={fake_s['contrast'].mean():.3f})", color="#c0392b")
axes[0, 1].set_title("B. Image Contrast (Luma Std) Distribution", fontweight="bold")
axes[0, 1].set_xlabel("Luminance Std Dev")
axes[0, 1].set_ylabel("Density")
axes[0, 1].legend()

# Sharpness (log10 Laplacian variance)
r_log_sharp = np.log10(np.clip(real_s["sharpness_lap_var"], 1e-7, None))
f_log_sharp = np.log10(np.clip(fake_s["sharpness_lap_var"], 1e-7, None))
axes[1, 0].hist(r_log_sharp, bins=40, alpha=0.55, density=True, label=f"Real (med={real_s['sharpness_lap_var'].median():.4f})", color="#1f7a4c")
axes[1, 0].hist(f_log_sharp, bins=40, alpha=0.55, density=True, label=f"Fake (med={fake_s['sharpness_lap_var'].median():.4f})", color="#c0392b")
axes[1, 0].set_title("C. Sharpness / Blur Score (log10 Laplacian Variance)", fontweight="bold")
axes[1, 0].set_xlabel("log10(Laplacian Variance)  [Left = Blurrier, Right = Sharper]")
axes[1, 0].set_ylabel("Density")
axes[1, 0].legend()

# Mean RGB channels
channels = ["Red", "Green", "Blue"]
r_means = [real_s["r_mean"].mean(), real_s["g_mean"].mean(), real_s["b_mean"].mean()]
r_stds = [real_s["r_mean"].std(), real_s["g_mean"].std(), real_s["b_mean"].std()]
f_means = [fake_s["r_mean"].mean(), fake_s["g_mean"].mean(), fake_s["b_mean"].mean()]
f_stds = [fake_s["r_mean"].std(), fake_s["g_mean"].std(), fake_s["b_mean"].std()]
cx = np.arange(3)
axes[1, 1].bar(cx - 0.18, r_means, yerr=r_stds, width=0.35, capsize=4, label="Real", color="#1f7a4c", alpha=0.85)
axes[1, 1].bar(cx + 0.18, f_means, yerr=f_stds, width=0.35, capsize=4, label="Fake", color="#c0392b", alpha=0.85)
axes[1, 1].set_xticks(cx)
axes[1, 1].set_xticklabels(channels)
axes[1, 1].set_ylim(0, 0.85)
axes[1, 1].set_title("D. Mean RGB Channel Intensities (±1 SD)", fontweight="bold")
axes[1, 1].set_ylabel("Normalized Intensity [0, 1]")
axes[1, 1].legend()

plt.suptitle("DS0001 Pixel & Image Quality Statistics — Stratified Sample (n=4,000)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig04_path = FIG_DIR / "fig04_pixel_brightness_blur_stats.png"
plt.savefig(fig04_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig04_path.name)


In [ ]:
# 6. FIG05 — Average 2D FFT Log-Magnitude Spectrum & 1D Azimuthal Radial Profile
mean_fft_real = fft_accum["real"] / max(fft_n["real"], 1)
mean_fft_sg2 = fft_accum["stylegan2"] / max(fft_n["stylegan2"], 1)
mean_fft_sd = fft_accum["stable_diffusion"] / max(fft_n["stable_diffusion"], 1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), dpi=300)
vmin = min(mean_fft_real.min(), mean_fft_sg2.min(), mean_fft_sd.min())
vmax = max(mean_fft_real.max(), mean_fft_sg2.max(), mean_fft_sd.max())

for ax, mat, title in [
    (axes[0], mean_fft_real, f"Mean 2D FFT: Real (n={fft_n['real']})"),
    (axes[1], mean_fft_sg2, f"Mean 2D FFT: StyleGAN2 (n={fft_n['stylegan2']})"),
    (axes[2], mean_fft_sd, f"Mean 2D FFT: Stable Diffusion (n={fft_n['stable_diffusion']})"),
]:
    im = ax.imshow(mat, cmap="inferno", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=9.5, fontweight="bold")
    ax.axis("off")

# 1D radial profile comparison
radii = np.arange(1, MAX_R)
palette = {
    "Real (n=2000)": ("#1f7a4c", "-", 2.2),
    "All Fakes (n=2000)": ("#c0392b", "--", 2.0),
    "StyleGAN2": ("#2980b9", "-.", 1.8),
    "Stable Diffusion": ("#8e44ad", ":", 2.0),
}
for name, (col, ls, lw) in palette.items():
    prof = np.mean(radial_by_group[name], axis=0)[1:]
    axes[3].plot(radii, prof, label=name, color=col, linestyle=ls, linewidth=lw)
axes[3].set_title("1D Azimuthal Radial Power Profile", fontsize=9.5, fontweight="bold")
axes[3].set_xlabel("Spatial Frequency Radius (px)")
axes[3].set_ylabel("Mean log(1 + |FFT|)")
axes[3].legend(fontsize=8)

plt.suptitle("DS0001 Frequency-Domain Forensics — Spectral Differences Between Real & AI Faces", fontsize=12.5, fontweight="bold")
plt.tight_layout()
fig05_path = FIG_DIR / "fig05_average_fft_spectrum.png"
plt.savefig(fig05_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig05_path.name)


In [ ]:
# 7. FIG06 (STEP-006) — Preprocessing & Strictly Training-Only Augmentation Preview
sample_real_row = df_split[(df_split["split"] == "train") & (df_split["label"] == 0)].iloc[10]
sample_fake_row = df_split[(df_split["split"] == "train") & (df_split["generator"] == "stable_diffusion")].iloc[10]

tf_resize = T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC)
tf_flip = T.Compose([tf_resize, T.RandomHorizontalFlip(p=1.0)])
tf_jpeg = T.Compose([tf_resize, RandomJPEGCompression(quality_min=42, quality_max=45, p=1.0)])
tf_blur = T.Compose([tf_resize, T.GaussianBlur(kernel_size=3, sigma=(1.4, 1.4))])
tf_robust = get_train_transforms_robust(img_size=224)

fig, axes = plt.subplots(2, 6, figsize=(15, 5.6), dpi=300)
col_titles = [
    "1. Original\n(200x200)",
    "2. Eval Resize\n(224x224, Det.)",
    "3. Train Only:\nHorizontal Flip",
    "4. Train Only:\nJPEG (q~45)",
    "5. Train Only:\nGaussian Blur",
    "6. Train Only:\nFull Robust Aug",
]

for r_idx, row in enumerate([sample_real_row, sample_fake_row]):
    with Image.open(DATA_ROOT / row["rel_path"]) as im:
        orig = im.convert("RGB")
    random.seed(SEED + r_idx)
    torch.manual_seed(SEED + r_idx)
    robust_img = denormalize_tensor(tf_robust(orig)).permute(1, 2, 0).numpy()

    imgs = [orig, tf_resize(orig), tf_flip(orig), tf_jpeg(orig), tf_blur(orig), robust_img]
    row_lbl = f"REAL ({row['image_id']})" if row["label"] == 0 else f"FAKE: {row['generator']}"
    for c_idx, disp in enumerate(imgs):
        ax = axes[r_idx, c_idx]
        ax.imshow(disp)
        if r_idx == 0:
            ax.set_title(col_titles[c_idx], fontsize=9, fontweight="bold")
        if c_idx == 0:
            ax.set_ylabel(row_lbl, fontsize=9, fontweight="bold")
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            ax.axis("off")

plt.suptitle("STEP-006 Preprocessing & Training-Only Augmentation Pipeline Verification", fontsize=12.5, fontweight="bold")
plt.tight_layout()
fig06_path = FIG_DIR / "fig06_augmentation_preview.png"
plt.savefig(fig06_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig06_path.name)


In [ ]:
# 8. Save Real EDA Summary JSON + Auto-Populated EDA Report MD
eda_summary = {
    "dataset_id": "DS0001",
    "built_utc": datetime.now(timezone.utc).isoformat(),
    "total_images": int(len(df_split)),
    "stratified_pixel_sample_n": int(len(df_stats)),
    "seed": SEED,
    "real_stats": {
        "brightness_mean": round(float(real_s["brightness"].mean()), 4),
        "brightness_std": round(float(real_s["brightness"].std()), 4),
        "contrast_mean": round(float(real_s["contrast"].mean()), 4),
        "contrast_std": round(float(real_s["contrast"].std()), 4),
        "sharpness_lap_var_median": round(float(real_s["sharpness_lap_var"].median()), 6),
        "sharpness_lap_var_mean": round(float(real_s["sharpness_lap_var"].mean()), 6),
        "rgb_mean": [round(float(x), 4) for x in r_means],
        "rgb_std": [round(float(x), 4) for x in r_stds],
    },
    "fake_stats": {
        "brightness_mean": round(float(fake_s["brightness"].mean()), 4),
        "brightness_std": round(float(fake_s["brightness"].std()), 4),
        "contrast_mean": round(float(fake_s["contrast"].mean()), 4),
        "contrast_std": round(float(fake_s["contrast"].std()), 4),
        "sharpness_lap_var_median": round(float(fake_s["sharpness_lap_var"].median()), 6),
        "sharpness_lap_var_mean": round(float(fake_s["sharpness_lap_var"].mean()), 6),
        "rgb_mean": [round(float(x), 4) for x in f_means],
        "rgb_std": [round(float(x), 4) for x in f_stds],
    },
    "high_freq_radial_tail_mean_r70_99": {
        "real": round(float(np.mean(radial_by_group["Real (n=2000)"], axis=0)[70:].mean()), 4),
        "all_fake": round(float(np.mean(radial_by_group["All Fakes (n=2000)"], axis=0)[70:].mean()), 4),
        "stylegan2": round(float(np.mean(radial_by_group["StyleGAN2"], axis=0)[70:].mean()), 4),
        "stable_diffusion": round(float(np.mean(radial_by_group["Stable Diffusion"], axis=0)[70:].mean()), 4),
    },
}

(REPO_DIR / "02_data/eda_summary.json").write_text(json.dumps(eda_summary, indent=2), encoding="utf-8")

rs = eda_summary["real_stats"]
fs = eda_summary["fake_stats"]
hf = eda_summary["high_freq_radial_tail_mean_r70_99"]

eda_report_md = f"""# Visual Exploratory Data Analysis Report — `DS0001` (`STEP-004`)

> **Owner:** Member 2 — Khadijatul Kobra Keya (`0242310005101223`)  
> **Source Notebook:** `04_notebooks/02_visual_eda.ipynb`  
> **Stratified Pixel/FFT Sample:** `n = {len(df_stats):,}` (`2,000` Real + `2,000` Fake balanced across all 13 generators, `SEED = 42`)  

## 1. Class Balance & Split Integrity (`fig01`)
- Total verified images: **50,000** (`25,000` Real / `25,000` AI-Generated Fake — exact `50.0% / 50.0%` balance).
- Synthetic faces are near-uniformly distributed across **13 distinct generators** (`1,923` each; `stylegan2` = `1,924`).
- Frozen leak-free splits (`ds0001_split_v1.csv`): **35,000 Train (70%) / 7,500 Val (15%) / 7,500 Test (15%)** with 0 overlapping `image_id`, `rel_path`, or `md5_hash`.

## 2. Pixel, Lighting, Contrast & Sharpness Statistics (`fig03`, `fig04`)

| Metric (n=4,000 Stratified Sample) | Real Faces | AI-Generated Faces |
|---|---:|---:|
| **Mean Brightness (Luminance)** | `{rs['brightness_mean']:.4f} ± {rs['brightness_std']:.4f}` | `{fs['brightness_mean']:.4f} ± {fs['brightness_std']:.4f}` |
| **Mean Contrast (Luminance Std)** | `{rs['contrast_mean']:.4f} ± {rs['contrast_std']:.4f}` | `{fs['contrast_mean']:.4f} ± {fs['contrast_std']:.4f}` |
| **Median Sharpness (Laplacian Var)** | `{rs['sharpness_lap_var_median']:.6f}` | `{fs['sharpness_lap_var_median']:.6f}` |
| **Mean RGB Channels** | `{rs['rgb_mean']}` | `{fs['rgb_mean']}` |

- **Environmental Variation (`fig03`):** Real and synthetic faces span dark/high-key lighting, soft-focus blur vs. sharp detail, varying contrast, and warm/cool illumination casts — confirming that simple brightness/color heuristics alone cannot separate classes cleanly.

## 3. Frequency-Domain Forensics (`fig05`)
- High-frequency radial power (`radius 70–99 px`, mean `log(1 + |FFT|)`):
  - **Real:** `{hf['real']:.4f}` | **All Fakes:** `{hf['all_fake']:.4f}` | **StyleGAN2:** `{hf['stylegan2']:.4f}` | **Stable Diffusion:** `{hf['stable_diffusion']:.4f}`
- Visible azimuthal and spectral roll-off differences between real camera portraits and GAN/Diffusion generators motivate our **Family 1 Traditional CV Baseline (`M1`: 2D FFT + LBP + Color → SVM/RF)** and our **Robustness Augmentation (`STEP-012`)**.

## 4. Preprocessing & Training-Only Augmentation Verification (`fig06`)
- Validation/test images undergo strictly deterministic `224×224` bicubic resizing and ImageNet normalization.
- Training-only robustness augmentations (horizontal flip, JPEG recompression `q=40–95`, Gaussian blur, color jitter) are verified visually in `fig06_augmentation_preview.png` and programmatically via `03_preprocessing/test_transforms.py`.
"""
(REPO_DIR / "02_data/eda_report.md").write_text(eda_report_md, encoding="utf-8")
print("Saved eda_summary.json and eda_report.md.")


In [ ]:
# Run full pytest suite on Kaggle (including PyTorch/torchvision determinism tests!)
run_masked(["pytest", "-v", "03_preprocessing/test_transforms.py"], cwd=REPO_DIR, secret=TOKEN)

# Register FIG01–FIG06, Update PROGRESS_TRACKER (GATE 2 PASS) & Auto-Push to GitHub
fig_reg_path = REPO_DIR / "06_evaluation/figure_registry.csv"
fig_df = pd.read_csv(fig_reg_path)
existing_ids = set(fig_df["fig_id"].astype(str))

new_figs = [
    ("FIG01", "06_evaluation/figures/eda/fig01_class_generator_distribution.png", "STEP-004", "Section 7", "DS0001 leak-free 70/15/15 split balance and 13-generator distribution", "04_notebooks/02_visual_eda.ipynb"),
    ("FIG02", "06_evaluation/figures/eda/fig02_sample_grid_real_vs_fake.png", "STEP-004", "Section 7", "4x4 visual sample gallery comparing Real faces against 12 synthetic generators", "04_notebooks/02_visual_eda.ipynb"),
    ("FIG03", "06_evaluation/figures/eda/fig03_environmental_variations.png", "STEP-004", "Section 7", "Data-driven examples of lighting, sharpness/blur, contrast, and color-cast variations", "04_notebooks/02_visual_eda.ipynb"),
    ("FIG04", "06_evaluation/figures/eda/fig04_pixel_brightness_blur_stats.png", "STEP-004", "Section 7", "Brightness, contrast, Laplacian sharpness, and RGB channel statistics (n=4,000)", "04_notebooks/02_visual_eda.ipynb"),
    ("FIG05", "06_evaluation/figures/eda/fig05_average_fft_spectrum.png", "STEP-004", "Section 7", "Mean 2D FFT log-magnitude spectra and 1D azimuthal radial power profiles", "04_notebooks/02_visual_eda.ipynb"),
    ("FIG06", "06_evaluation/figures/eda/fig06_augmentation_preview.png", "STEP-006", "Section 8", "Preprocessing and strictly training-only robustness augmentation preview", "04_notebooks/02_visual_eda.ipynb"),
]
rows_to_add = [
    {"fig_id": fid, "filename": fn, "step_id": sid, "report_section": sec, "caption": cap, "source_script": src}
    for (fid, fn, sid, sec, cap, src) in new_figs if fid not in existing_ids
]
if rows_to_add:
    pd.concat([fig_df, pd.DataFrame(rows_to_add)], ignore_index=True).to_csv(fig_reg_path, index=False)

# Update PROGRESS_TRACKER.md: STEP-004 DONE, STEP-006 DONE, GATE 2 PASS
tracker_path = REPO_DIR / "PROGRESS_TRACKER.md"
today_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
updated = []
for line in tracker_path.read_text(encoding="utf-8").splitlines():
    if line.startswith("| `STEP-004` |"):
        line = (
            f"| `STEP-004` | M3 (Due **Fri, 27 Nov**) | Visual EDA: Distributions, Image Stats & Sample Grids "
            f"| M2 Khadijatul Kobra Keya | Nov 26–27 | `DONE` | `fig01`–`fig05`, `eda_summary.json`, `eda_report.md` pushed from Kaggle ({today_str}) |"
        )
    elif line.startswith("| `STEP-006` |"):
        line = (
            f"| `STEP-006` | M3 (Due **Fri, 27 Nov**) | Preprocessing & Training-Only Augmentation Pipeline "
            f"| M2 Khadijatul Kobra Keya | Nov 27 | `DONE` | `transforms.py`, `dataset.py`, pytest PASS + `fig06` ({today_str}) |"
        )
    elif "**GATE 2 (After `STEP-006`):**" in line:
        line = f"- [x] **GATE 2 (After `STEP-006`):** `DS0001` verified, visual EDA complete, zero-leakage split locked, training-only augmentation verified. *(PASS — {today_str})*"
    updated.append(line)
tracker_path.write_text("\n".join(updated) + "\n", encoding="utf-8")

run_masked(["git", "add", "-A"], cwd=REPO_DIR, secret=TOKEN)
status_out = run_masked(["git", "status", "--porcelain"], cwd=REPO_DIR, secret=TOKEN)
if status_out.strip():
    run_masked(
        ["git", "commit", "-m", "eda(step-004,step-006): push visual EDA fig01-fig06, eda_report, and GATE 2 PASS from Kaggle"],
        cwd=REPO_DIR,
        secret=TOKEN,
    )
    run_masked(["git", "push", AUTH_URL, f"HEAD:refs/heads/{BRANCH}"], cwd=REPO_DIR, secret=TOKEN)
    sha = run_masked(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, secret=TOKEN)
    print(f"\n[SUCCESS] STEP-004 & STEP-006 outputs pushed to GitHub main ({sha}) | GATE 2 PASS!")
else:
    print("\n[INFO] Nothing new to commit.")

print("\n=== Ready-to-Paste Google Sheet Milestone 3 Evidence URLs ===")
print("Evidence URL 1: https://github.com/isthisdeception/computer-vission-course-work/blob/main/02_data/eda_report.md")
print("Evidence URL 2: https://github.com/isthisdeception/computer-vission-course-work/blob/main/06_evaluation/figures/eda/fig06_augmentation_preview.png")
